In [0]:
%sql
-- ===== CONTAGEM TOTAL DE JOGOS =====
-- Conta o número total de jogos disponíveis na base de dados
--
-- Métricas:
--   • total_registros: número total de linhas na tabela
--   • jogos_unicos: número de jogos únicos (cada app_id conta uma vez)
--
-- Nota: Se um jogo tiver múltiplos gêneros, ele pode aparecer em várias
-- linhas após fazer EXPLODE, mas o app_id continua único.
SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT app_id) AS jogos_unicos
FROM workspace.default.steam_games;

In [0]:
%sql --name jogos_generos_limpos
-- ===== LIMPEZA DE DADOS: GÊNEROS =====
-- Esta célula prepara os dados de gêneros para análise posterior
--
-- Problema: A coluna "genres" contém um array JSON que pode ter dois formatos:
--   1) String simples: "Action", "Indie", "Casual"
--   2) Objeto JSON: {"id":"51","description":"Animation & Modeling"}
--
-- Solução:
--   • Faz o parsing do array JSON com from_json
--   • Usa EXPLODE para criar uma linha por gênero
--   • Extrai o campo "description" quando o formato é objeto JSON
--   • Mantém o valor direto quando é string simples
--
-- Resultado: Dataset limpo com uma linha por (jogo, gênero) pronto para agregações

SELECT 
    app_id,
    name,
    price,
    positive,
    negative,
    -- Trata os dois formatos de gênero:
    COALESCE(
        get_json_object(exploded_genre, '$.description'),  -- Extrai "description" se for objeto JSON
        exploded_genre  -- Usa o valor direto se for string simples
    ) AS genero
FROM workspace.default.steam_games
LATERAL VIEW EXPLODE(from_json(genres, 'array<string>')) AS exploded_genre
WHERE genres IS NOT NULL
  AND (positive + negative) > 0  -- Apenas jogos com avaliações

In [0]:
%sql
-- 1A) Existe diferença expressiva de aceitação do público entre jogos gratuitos e títulos pagos?
--
-- ===== ANÁLISE: MODELO DE MONETIZAÇÃO x APROVAÇÃO =====
-- Compara jogos gratuitos vs pagos em relação à recepção dos usuários
-- 
-- Lógica de classificação:
--   • Gratuito: quando price = 0 ou price_status = 'free'
--   • Pago: todos os demais jogos
--
-- Métricas calculadas:
--   • total_titulos: quantidade de jogos em cada categoria
--   • soma_positivas: total de avaliações positivas
--   • soma_negativas: total de avaliações negativas
--   • taxa_aprovacao_pct: percentual de avaliações positivas sobre o total
--
-- Filtro: considera apenas jogos que possuem pelo menos uma avaliação
SELECT 
    CASE 
        WHEN price = 0 OR LOWER(price_status) = 'free' THEN 'Gratuito'
        ELSE 'Pago'
    END AS modelo_monetizacao,
    COUNT(*) AS total_titulos,
    SUM(positive) AS soma_positivas,
    SUM(negative) AS soma_negativas,
    ROUND((SUM(positive) * 100.0) / NULLIF(SUM(positive) + SUM(negative), 0), 2) AS taxa_aprovacao_pct
FROM workspace.default.steam_games
WHERE (positive + negative) > 0
GROUP BY 1;

In [0]:
%sql
-- 2A) Qual a faixa de preço médio observada nas categorias de jogos mais bem avaliadas?
--
-- ===== ANÁLISE: PREÇO MÉDIO POR CATEGORIA (MELHORES AVALIADAS) =====
-- Identifica as categorias de jogos com melhor taxa de aprovação e analisa seus preços médios
--
-- Utiliza os dados limpos da célula anterior (jogos_generos_limpos)
--
-- Estratégia:
--   • Calcula taxa de aprovação: (positivas / total de avaliações) * 100
--   • Filtra apenas categorias com volume significativo (>= 50 jogos avaliados)
--   • Ordena pelas categorias mais bem avaliadas
--   • Calcula preço médio, mínimo e máximo de cada categoria
--
-- Métricas:
--   • genero: categoria/gênero do jogo
--   • total_jogos: quantidade de jogos na categoria
--   • taxa_aprovacao_pct: percentual de avaliações positivas
--   • preco_medio: preço médio dos jogos da categoria
--   • preco_minimo: menor preço encontrado
--   • preco_maximo: maior preço encontrado
--
-- Filtros aplicados:
--   • Jogos pagos (price > 0)
--   • Categorias com pelo menos 50 jogos
SELECT 
    genero,
    COUNT(*) AS total_jogos,
    ROUND((SUM(positive) * 100.0) / NULLIF(SUM(positive) + SUM(negative), 0), 2) AS taxa_aprovacao_pct,
    ROUND(AVG(price), 2) AS preco_medio,
    ROUND(MIN(price), 2) AS preco_minimo,
    ROUND(MAX(price), 2) AS preco_maximo
FROM jogos_generos_limpos
WHERE genero IS NOT NULL 
  AND LENGTH(genero) > 0
  AND price > 0  -- Considera apenas jogos pagos para análise de preço
GROUP BY genero
HAVING COUNT(*) >= 50  -- Filtra categorias com volume significativo
ORDER BY taxa_aprovacao_pct DESC
LIMIT 15;  -- Top 15 categorias mais bem avaliadas

In [0]:
%sql
-- 3A) Como evoluiu o volume anual de novos lançamentos?
--
-- ===== ANÁLISE: EVOLUÇÃO DO VOLUME DE LANÇAMENTOS POR ANO =====
-- Analisa a quantidade de jogos lançados a cada ano na plataforma Steam
--
-- Estratégia:
--   • Extrai o ano da coluna release_date
--   • Conta o número de jogos únicos lançados em cada ano
--   • Ordena cronologicamente para visualizar a tendência temporal
--
-- Métricas:
--   • ano_lancamento: ano extraído da data de lançamento
--   • total_lancamentos: quantidade de jogos lançados naquele ano
--   • jogos_pagos: quantidade de jogos pagos (price > 0)
--   • jogos_gratuitos: quantidade de jogos gratuitos (price = 0)
--
-- Filtros aplicados:
--   • Remove jogos sem data de lançamento
--   • Remove anos inválidos ou futuros (considera até 2026)
SELECT 
    YEAR(release_date) AS ano_lancamento,
    COUNT(DISTINCT app_id) AS total_lancamentos,
    COUNT(DISTINCT CASE WHEN price > 0 THEN app_id END) AS jogos_pagos,
    COUNT(DISTINCT CASE WHEN price = 0 THEN app_id END) AS jogos_gratuitos,
    ROUND(AVG(price), 2) AS preco_medio_lancamentos
FROM workspace.default.steam_games
WHERE release_date IS NOT NULL
  AND YEAR(release_date) >= 2012  -- Remove anos muito antigos
  AND YEAR(release_date) <= 2026  -- Remove anos futuros
GROUP BY YEAR(release_date)
ORDER BY ano_lancamento ASC;

In [0]:
%sql
-- ===== SILVER: TABELA PRINCIPAL DE JOGOS LIMPOS =====
-- Cria a tabela silver a nível de jogo (uma linha por jogo)
-- 
-- Transformações aplicadas:
--   • Classifica modelo de monetizacao (Gratuito vs Pago)
--   • Filtra apenas jogos com pelo menos uma avaliacao (qualidade)
--   • Filtra jogos sem data de lancamento
--   • Mantem colunas relevantes para analise downstream

CREATE TABLE IF NOT EXISTS workspace.default.steam_games_silver AS
SELECT 
    app_id,
    name,
    release_date,
    YEAR(release_date) AS ano_lancamento,
    price,
    price_status,
    CASE 
        WHEN price = 0 OR LOWER(price_status) = 'free' THEN 'Gratuito'
        ELSE 'Pago'
    END AS modelo_monetizacao,
    estimated_owners,
    genres,
    categories,
    positive,
    negative,
    positive + negative AS total_avaliacoes,
    ROUND((positive * 100.0) / NULLIF(positive + negative, 0), 2) AS taxa_aprovacao_pct,
    recommendations,
    peak_ccu,
    metacritic_score,
    user_score,
    average_playtime_forever,
    median_playtime_forever,
    achievements,
    dlc_count,
    windows,
    mac,
    linux,
    developers,
    publishers
FROM workspace.default.steam_games
WHERE (positive + negative) > 0  -- Apenas jogos com avaliacoes
  AND release_date IS NOT NULL;

-- Amostra dos dados criados na camada Silver
SELECT * FROM workspace.default.steam_games_silver LIMIT 10;

In [0]:
%sql
-- ===== SILVER: TABELA DE GÊNEROS NORMALIZADA =====
-- Cria a tabela silver com gêneros explodidos (uma linha por jogo-gênero)
--
-- Transformações aplicadas:
--   • Faz o parsing do array JSON de gêneros
--   • Usa EXPLODE para criar uma linha por gênero
--   • Trata os dois formatos (string simples e objeto JSON)
--   • Filtra gêneros vazios

CREATE TABLE IF NOT EXISTS workspace.default.steam_generos_silver AS
SELECT 
    s.app_id,
    s.name,
    s.price,
    s.modelo_monetizacao,
    s.positive,
    s.negative,
    s.total_avaliacoes,
    COALESCE(
        get_json_object(exploded_genre, '$.description'),
        exploded_genre
    ) AS genero
FROM workspace.default.steam_games_silver s
LATERAL VIEW EXPLODE(from_json(s.genres, 'array<string>')) AS exploded_genre
WHERE s.genres IS NOT NULL
  AND COALESCE(
        get_json_object(exploded_genre, '$.description'),
        exploded_genre
    ) IS NOT NULL
  AND LENGTH(COALESCE(
        get_json_object(exploded_genre, '$.description'),
        exploded_genre
    )) > 0;

-- Amostra dos dados criados na camada Silver
SELECT * FROM workspace.default.steam_generos_silver LIMIT 10;

In [0]:
%sql
-- ===== GOLD 1: MODELO DE MONETIZAÇÃO x APROVAÇÃO =====
-- Agrega a taxa de aprovação por modelo de monetizacao (Gratuito vs Pago)
--
-- Fonte: workspace.default.steam_games_silver
-- Responde: Existe diferenca de aceitacao entre jogos gratuitos e pagos?

CREATE TABLE IF NOT EXISTS workspace.default.steam_gold_monetizacao AS
SELECT 
    modelo_monetizacao,
    COUNT(*) AS total_titulos,
    SUM(positive) AS soma_positivas,
    SUM(negative) AS soma_negativas,
    ROUND((SUM(positive) * 100.0) / NULLIF(SUM(positive) + SUM(negative), 0), 2) AS taxa_aprovacao_pct
FROM workspace.default.steam_games_silver
GROUP BY modelo_monetizacao
ORDER BY modelo_monetizacao;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_monetizacao;

In [0]:
%sql
-- ===== GOLD 2: PREÇO MÉDIO POR GÊNERO (MELHORES AVALIADOS) =====
-- Agrega preco medio e taxa de aprovacao por genero
-- Filtra apenas generos com volume significativo (>= 50 jogos) e jogos pagos
--
-- Fonte: workspace.default.steam_generos_silver
-- Responde: Qual a faixa de preco medio nas categorias mais bem avaliadas?

CREATE TABLE IF NOT EXISTS workspace.default.steam_gold_preco_genero AS
SELECT 
    genero,
    COUNT(*) AS total_jogos,
    ROUND((SUM(positive) * 100.0) / NULLIF(SUM(positive) + SUM(negative), 0), 2) AS taxa_aprovacao_pct,
    ROUND(AVG(price), 2) AS preco_medio,
    ROUND(MIN(price), 2) AS preco_minimo,
    ROUND(MAX(price), 2) AS preco_maximo
FROM workspace.default.steam_generos_silver
WHERE price > 0  -- Apenas jogos pagos para analise de preco
GROUP BY genero
HAVING COUNT(*) >= 50  -- Volume significativo
ORDER BY taxa_aprovacao_pct DESC
LIMIT 15;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_preco_genero;

In [0]:
%sql
-- ===== GOLD 3: EVOLUÇÃO ANUAL DE LANÇAMENTOS =====
-- Agrega o volume de lancamentos por ano, separando pagos e gratuitos
--
-- Fonte: workspace.default.steam_games_silver
-- Responde: Como evoluiu o volume anual de novos lancamentos?

CREATE TABLE IF NOT EXISTS workspace.default.steam_gold_lancamentos AS
SELECT 
    ano_lancamento,
    COUNT(DISTINCT app_id) AS total_lancamentos,
    COUNT(DISTINCT CASE WHEN price > 0 THEN app_id END) AS jogos_pagos,
    COUNT(DISTINCT CASE WHEN price = 0 THEN app_id END) AS jogos_gratuitos,
    ROUND(AVG(price), 2) AS preco_medio_lancamentos
FROM workspace.default.steam_games_silver
WHERE ano_lancamento >= 2012
  AND ano_lancamento <= 2026
GROUP BY ano_lancamento
ORDER BY ano_lancamento ASC;

-- Resultado da camada Gold
SELECT * FROM workspace.default.steam_gold_lancamentos;